# Guided Lab: Ingest & Store a Document Set

Objective: load 5 local PDFs, process them using 3 chunking configurations, and embed them using a local Hugging Face CPU model.

This notebook follows the basic retrieval pipeline from the LangChain docs: load documents, split them into chunks, embed the chunks, and store them in a vector store. The official docs also note that Hugging Face embeddings can run locally on CPU and that local embeddings have lower latency than hosted APIs for short queries.


## Learning goals

By the end of this notebook, you should be able to:

1. Load 5 local PDF files into LangChain `Document` objects.
2. Add structured metadata to each document.
3. Split documents using 3 different chunking configurations.
4. Generate embeddings locally on CPU.
5. Store each chunk set in a local vector store.
6. Compare the chunk counts produced by each configuration.


## 1) Install packages

Run this once in a notebook cell:

```bash
pip install -U pypdf langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb sentence-transformers
```

If you later want retriever traces, add `langsmith` as well.


In [1]:
%pip install -qU pypdf langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb sentence-transformers langsmith


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2) Prepare your PDF folder

Place 5 local PDFs in a folder named `./pdfs`.

Example:

```text
pdfs/
├── doc1.pdf
├── doc2.pdf
├── doc3.pdf
├── doc4.pdf
└── doc5.pdf
```

The LangChain PDF loader docs show that PDF files are one of the standard document ingestion sources. citeturn543870search2turn543870search8


In [ ]:
from pathlib import Path
from typing import List
from datetime import datetime

from langchain_core.documents import Document

PDF_DIR = Path('./pdfs')
pdf_files = sorted(PDF_DIR.glob('*.pdf'))

print('PDF folder exists:', PDF_DIR.exists())
print('PDF count:', len(pdf_files))
for p in pdf_files:
    print(' -', p.name)


## 3) Load PDFs

This helper loads each PDF into a list of `Document` objects and attaches basic metadata such as file name and page number.


In [ ]:
from pypdf import PdfReader

def load_pdf(path: Path) -> List[Document]:
    reader = PdfReader(str(path))
    docs: List[Document] = []
    for page_num, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ''
        if text.strip():
            docs.append(
                Document(
                    page_content=text,
                    metadata={
                        'source': str(path),
                        'file_name': path.name,
                        'type': 'pdf',
                        'page': page_num,
                        'ingested_at': datetime.utcnow().isoformat(),
                    },
                )
            )
    return docs

all_docs: List[Document] = []
for pdf in pdf_files:
    all_docs.extend(load_pdf(pdf))

print('Loaded documents:', len(all_docs))
if all_docs:
    print('Example metadata:', all_docs[0].metadata)


## 4) Add a small metadata helper

Metadata helps later when you filter or inspect retrieved chunks.

A simple metadata set is enough for a basic lab:

- `source`
- `file_name`
- `page`
- `type`
- `chunk_config`


In [ ]:
def add_chunk_metadata(docs: List[Document], chunk_config: str) -> List[Document]:
    enriched = []
    for doc in docs:
        meta = dict(doc.metadata)
        meta['chunk_config'] = chunk_config
        enriched.append(Document(page_content=doc.page_content, metadata=meta))
    return enriched


## 5) Define 3 chunking configurations

To keep the lab basic, we will use three different character-based chunking setups:

- small chunks
- medium chunks
- large chunks

The LangChain docs explain that splitters break large documents into smaller chunks so they fit retrieval and context-window limits.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_configs = {
    'small': RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50),
    'medium': RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100),
    'large': RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150),
}

chunk_configs


## 6) Split documents with each configuration

We will generate three chunk sets and compare their sizes.


In [ ]:
chunk_sets = {}
chunk_stats = []

for name, splitter in chunk_configs.items():
    split_docs = splitter.split_documents(all_docs)
    split_docs = add_chunk_metadata(split_docs, name)
    chunk_sets[name] = split_docs
    chunk_stats.append((name, len(split_docs)))

chunk_stats


In [ ]:
for name, docs in chunk_sets.items():
    print(f'\n=== {name.upper()} ===')
    print('Chunks:', len(docs))
    if docs:
        print('Sample metadata:', docs[0].metadata)
        print('Sample text preview:', docs[0].page_content[:300].replace('\n', ' '))


## 7) Create a local CPU embedding model

LangChain’s Hugging Face docs say `HuggingFaceEmbeddings` can run open-source embedding models locally. The embeddings docs also note that local CPU models can be suitable for short queries and that hosted APIs introduce network latency.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

print('Embedding model ready on CPU.')


## 8) Store each chunk set in Chroma

We will save each chunk configuration into a separate local collection directory.

This keeps the results easy to compare and inspect.


In [ ]:
from langchain_chroma import Chroma

base_dir = Path('./chroma_labs')
base_dir.mkdir(exist_ok=True)

stores = {}

for name, docs in chunk_sets.items():
    persist_dir = base_dir / name
    store = Chroma(
        collection_name=f'pdf_lab_{name}',
        embedding_function=embeddings,
        persist_directory=str(persist_dir),
    )
    store.add_documents(docs)
    stores[name] = store
    print(f'Saved {len(docs)} chunks to {persist_dir}')


## 9) Compare retrieval from each configuration

We can now test how each chunk size behaves with a sample question.


In [ ]:
query = 'What are the main topics covered in the PDFs?'

for name, store in stores.items():
    print(f'\n--- {name.upper()} ---')
    results = store.similarity_search(query, k=2)
    for i, doc in enumerate(results, 1):
        print(f'Result {i}:', doc.metadata.get('file_name'), 'page', doc.metadata.get('page'))
        print(doc.page_content[:250].replace('\n', ' '))
        print()


## 10) Inspect embedding latency

The LangChain embeddings docs explain that local CPU models reduce network round trips but still take time to run. This cell times a small batch so you can get a rough feel for local embedding cost.


In [ ]:
import time

sample_texts = [doc.page_content for doc in all_docs[:5]]

start = time.perf_counter()
vectors = embeddings.embed_documents(sample_texts)
elapsed = time.perf_counter() - start

print('Embedded texts:', len(vectors))
print(f'Elapsed seconds: {elapsed:.3f}')
print('Vector dimension:', len(vectors[0]) if vectors else 0)


## 11) Basic summary table

This helps compare the three chunking setups at a glance.


In [ ]:
summary = []
for name, docs in chunk_sets.items():
    summary.append({
        'config': name,
        'chunks': len(docs),
        'persist_dir': str(base_dir / name),
    })

summary


## 12) Clean-up helper

Use this when you want to remove the local vector-store directories and start over.


In [ ]:
import shutil

def cleanup_lab():
    if base_dir.exists():
        shutil.rmtree(base_dir)
        print('Removed', base_dir)
    else:
        print('Nothing to clean')

# cleanup_lab()


## 13) What to try next

Once this lab works end-to-end, you can extend it later with:

- retriever queries
- metadata filtering
- reranking
- evaluation
- LangSmith retriever traces

LangChain’s knowledge-base docs describe loading, splitting, embedding, and storing as the core retrieval building blocks.


## Key takeaways

- PDFs are loaded into standardized `Document` objects.
- Metadata should be attached during ingestion.
- Chunk size changes retrieval behavior.
- Local Hugging Face embeddings can run on CPU.
- Chroma can persist vector data locally.
- Comparing multiple chunking setups is a good first lab before moving into deeper RAG work.


## References

- PDF loaders: https://docs.langchain.com/oss/python/integrations/document_loaders/index#pdfs
- Seeding the vector store: https://docs.langchain.com/oss/python/langchain/knowledge-base#seeding-the-vector-store
- Splitting documents: https://docs.langchain.com/oss/python/langchain/rag#splitting-documents
- Hugging Face embeddings: https://docs.langchain.com/oss/python/integrations/providers/huggingface#huggingfaceembeddings
- Embeddings latency: https://docs.langchain.com/oss/python/integrations/embeddings/index#latency
